In [8]:
!pip install numpy
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
!pip install seaborn
!pip install torch

  Using cached pytorch-1.0.2.tar.gz (689 bytes)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pytorch (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [37 lines of output]
      Traceback (most recent call last):
        File "/Users/vishnusai/Downloads/LSTM_GRU_TEMPERATURE_FORECASTING/.venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/Users/vishnusai/Downloads/LSTM_GRU_TEMPERATURE_FORECASTING/.venv/lib/python3.14/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/Users/vishnusai/Downloads/LSTM_GRU_TEMPERATURE_FORECASTING/.venv/lib

In [7]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

train_df = pd.read_csv('https://s3.ap-south-1.amazonaws.com/new-assets.ccbp.in/frontend/content/aiml/classical-ml/weather_train.csv')
test_df = pd.read_csv('https://s3.ap-south-1.amazonaws.com/new-assets.ccbp.in/frontend/content/aiml/classical-ml/weather_test.csv')

WINDOW_SIZE = 24

# Converts flat rows into sequences for LSTM/GRU
# Each sample: 24 consecutive hours as input, next hour's temperature as target
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i : i + window_size])
        y.append(data[i + window_size, 0])  # 0 = temperature column
    return np.array(X), np.array(y)

print(f'Train: {train_df.shape}')
print(f'Test: {test_df.shape}')
train_df.head()

ModuleNotFoundError: No module named 'torch'

In [8]:
train_df = train_df.drop(columns=['datetime'])

In [9]:
train_df

,temperature,humidity,pressure
0,2.48,95.97,985.44
1,2.89,95.63,985.57
2,3.26,94.97,985.52
3,3.44,95.48,985.54
4,3.58,97.72,985.52
...,...,...,...
35003,-0.19,98.22,1005.68
35004,-0.32,98.60,1005.68
35005,0.08,98.97,1005.47
35006,0.24,98.95,1005.30


In [10]:
feature_cols =train_df.select_dtypes(include = [np.number]).columns 
scaler = MinMaxScaler(feature_range = (0,1)) 
train_scaled = scaler.fit_transform(train_df[feature_cols]) 
test_scaled = scaler.transform(test_df[feature_cols])
print(f'Train scaled: {train_scaled.shape}')
print(f'Test scaled: {test_scaled.shape}')

Train scaled: (35008, 3)
Test scaled: (8753, 3)


In [11]:
x_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)
x_test,_ = create_sequences(test_scaled, WINDOW_SIZE)

In [12]:
class WeatherLSTM(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
       
        self.lstm = nn.LSTM(input_size = input_size, hidden_size = hidden_size, batch_first = True)
        self.fc = nn.Linear(hidden_size, 1)  # Assuming we want to predict a single value (e.g., temperature)
    def forward(self,x):
        out,hidden = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [13]:
import time

In [14]:
def train_model(model, x_train , y_train, batch_size = 16, lr = 0.003, epochs = 100, print_every = 10 ):
    device = next(model.parameters()).device # Get the device of the model (CPU or GPU)
    x_train_tensor = torch.tensor(x_train , dtype = torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype = torch.float32).reshape(-1,1)
    dataset = TensorDataset(x_train_tensor, y_train_tensor)
    data_loader = DataLoader(dataset, batch_size = batch_size, shuffle = True) 
    optimizer = torch.optim.Adam(model.parameters(), lr = lr)
    loss_fn = nn.MSELoss()

    loss_history = []
    model.train()
    start_time = time.time()
    for epoch in range(1, epochs + 1):
        epoch_loss = 0 
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device)
            outputs = model(x)
            loss = loss_fn(outputs, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm = 1.0) # checks the overall gradient norm. If it exceeds 1.0, scale the gradients down so the overall norm becomes 1.0
            optimizer.step() # it is used to update model weights and bias using the gradients calculated during backpropogation 
            epoch_loss += loss.item() * x.size(0) 
        avg_loss = epoch_loss / len(dataset)
        loss_history.append(avg_loss)
        if epoch % print_every == 0:
            print(f'Epoch [{epoch}/{epochs}], Loss: {avg_loss:.4f}')
    training_time = time.time() - start_time 
    print(f"Training time: {training_time:.2f} seconds")
    return loss_history, training_time  




In [15]:
feature_cols

Index(['temperature', 'humidity', 'pressure'], dtype='str')

In [16]:
device = torch.device(
    'mps' if torch.backends.mps.is_available() else 'cpu'
)

print(f"Using Device: {device}")

HIDDEN_SIZE = 500 
lstm_model = WeatherLSTM(input_size  =  x_train.shape[2], hidden_size = HIDDEN_SIZE).to(device )


Using Device: mps


In [ ]:
print("Training LSTM.....")
lstm_loss, lstm_time = train_model(lstm_model, x_train , y_train)

In [ ]:
torch.save(lstm_model.state_dict(), "weather_lstm.pth")

print("LSTM model saved successfully!")

LSTM model saved successfully!


In [ ]:
print("Training GRU.....")

In [17]:
device = torch.device(
    'mps' if torch.backends.mps.is_available() else 'cpu'
)

print(f"Using Device: {device}")

Using Device: mps


In [18]:
def predict_model(model, x_test):
    model.eval() 
    device = next(model.parameters()).device 
    x_test_tensor = torch.tensor(x_test, dtype = torch.float32).to(device)
    with torch.no_grad():
        predictions = model(x_test_tensor)

    return predictions.cpu().numpy()


In [19]:
lstm_predictions = predict_model(lstm_model, x_test)
print(lstm_predictions[:10])

[[0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]
 [0.03700909]]


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG14GFamilyCommandBuffer: 0x137c2b8f0>
    label = <none> 
    device = <AGXG14GDevice: 0x125362c00>
        name = Apple M2 
    commandQueue = <AGXG14GFamilyCommandQueue: 0x124c8b000>
        label = <none> 
        device = <AGXG14GDevice: 0x125362c00>
            name = Apple M2 
    retainedReferences = 1
